# Notebook 03 — Multi-Head Attention

**Goal:** Understand why transformers use multiple attention heads, and how the tensors are split and recombined.

Instead of one attention operation over the whole embedding vector, we split the embedding into smaller subspaces:

```
X  [batch, seq_len, d_model]
   │
   ▼  linear projections
Q, K, V  [batch, seq_len, d_model]
   │
   ▼  split into heads
[batch, n_heads, seq_len, d_k]
   │
   ▼  attention in parallel
[batch, n_heads, seq_len, d_k]
   │
   ▼  merge heads
[batch, seq_len, d_model]
```

---

### Contents
1. [Why multiple heads?](#1)
2. [Split and merge](#2)
3. [A minimal MultiHeadAttention module](#3)
4. [Per-head attention patterns](#4)
5. [Key takeaways](#5)


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('=' * 60)
print('  Notebook 03 — Multi-Head Attention')
print('=' * 60)
print(f'  PyTorch : {torch.__version__}')
print('=' * 60)


<a id='1'></a>
## 1 — Why multiple heads?

A single attention head gives one attention pattern.

Multiple heads let the model learn **different views of the same sequence** at the same time. In a trained model, different heads often specialise in things like:

- local neighbours
- sentence-start tokens
- syntactic relationships
- semantic similarity

The key point is that the model gets **multiple parallel subspaces** to work with.


<a id='2'></a>
## 2 — Split and merge

We'll use a small example with:
- `d_model = 8`
- `n_heads = 2`
- so each head gets `d_k = 4`


In [ ]:
batch_size = 1
seq_len = 4
d_model = 8
n_heads = 2
d_k = d_model // n_heads

x = torch.randn(batch_size, seq_len, d_model)

print(f'Input shape: {tuple(x.shape)}')
print(f'd_model     : {d_model}')
print(f'n_heads     : {n_heads}')
print(f'd_k         : {d_k}')


In [ ]:
def split_heads(x, n_heads):
    batch, seq, d_model = x.shape
    d_k = d_model // n_heads
    return x.view(batch, seq, n_heads, d_k).transpose(1, 2)


def merge_heads(x):
    batch, n_heads, seq, d_k = x.shape
    return x.transpose(1, 2).contiguous().view(batch, seq, n_heads * d_k)

x_heads = split_heads(x, n_heads)
x_merged = merge_heads(x_heads)

print('After split :', tuple(x_heads.shape), ' -> (batch, heads, seq, d_k)')
print('After merge :', tuple(x_merged.shape), ' -> back to (batch, seq, d_model)')
print('Round-trip match:', torch.allclose(x, x_merged))


<a id='3'></a>
## 3 — A minimal MultiHeadAttention module


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.last_attn_weights = None

    def forward(self, x, mask=None):
        batch, seq, _ = x.shape

        Q = self.W_q(x).view(batch, seq, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch, seq, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch, seq, self.n_heads, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        weights = torch.nan_to_num(weights, nan=0.0)
        self.last_attn_weights = weights.detach()

        out = weights @ V
        out = out.transpose(1, 2).contiguous().view(batch, seq, self.d_model)
        return self.W_o(out)


In [ ]:
tokens = ['The', 'robot', 'picked', 'up']
mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
mask = mask.unsqueeze(0).unsqueeze(0)

mha = MultiHeadAttention(d_model=8, n_heads=2)
out = mha(x, mask=mask)

print('Output shape           :', tuple(out.shape))
print('Attention weights shape:', tuple(mha.last_attn_weights.shape))


<a id='4'></a>
## 4 — Per-head attention patterns

A useful visual here is the attention heatmap **per head**. That's where multi-head attention becomes concrete.


In [ ]:
weights = mha.last_attn_weights[0]

fig, axes = plt.subplots(1, n_heads, figsize=(10, 4))
if n_heads == 1:
    axes = [axes]

for h, ax in enumerate(axes):
    sns.heatmap(weights[h].numpy(), annot=True, fmt='.2f', cmap='Blues',
                xticklabels=tokens, yticklabels=tokens,
                linewidths=0.4, linecolor='white', square=True, ax=ax)
    ax.set_title(f'Head {h}', fontweight='bold')
    ax.set_xlabel('Key token')
    ax.set_ylabel('Query token')

plt.tight_layout()
plt.show()


<a id='5'></a>
## 5 — Key takeaways

- Multi-head attention splits `d_model` into smaller per-head subspaces
- Each head runs scaled dot-product attention independently
- The outputs are concatenated and projected back to `d_model`
- The most useful way to inspect it is to compare **attention matrices head by head**

Next, we'll place multi-head attention inside a full transformer block.
